# Record Linkage Model Implementation

This notebook implements the deterministic record linkage algorithm from the SQL model and validates it against the synthetic patient dataset with ground truth labels.

## Matching Strategy (Cascading)
1. **EGK + Date of Birth** — Match on insurance number + birth date
2. **EGK + Full Name** — Match on insurance number + first name + last name  
3. **Name + DOB + PLZ** — Match on full name + birth date + postal code

Unmatched records after all phases are grouped using **transitive closure** via Union-Find.


In [1]:
import pandas as pd
from collections import defaultdict

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)


## 1. Load Dataset


In [2]:
df = pd.read_csv('../../data/patients_gen.csv', dtype=str)
df['entity_id'] = df['entity_id'].astype(int)

print(f"Total records: {len(df):,}")
print(f"Unique entities (ground truth): {df['entity_id'].nunique():,}")
print(f"\nSample records:")
df.head(10)


Total records: 500,505
Unique entities (ground truth): 100,000

Sample records:


,entity_id,DWH_ZEITRAUM,Datenkoerper,EGKVERSICHERTENNUMMER,VORNAME,NACHNAME,Geburtsdatum,PLZ,GESCHLECHT,ERGEBNISONLINEPRUEFUNG,ERRORCODE,ONLINEPRUEFUNGSDATUM,golden_EGKVERSICHERTENNUMMER,golden_VORNAME,golden_NACHNAME,golden_Geburtsdatum,golden_PLZ,golden_GESCHLECHT
0,98713,20241,UNBEARBEITET,P587975846,ELISABETH,Kirchhoff,1985-05-29,48899,M,NaN,NaN,NaN,P587975846,ELISABETH,Kirchhoff,1985-05-29,47777,M
1,40447,20232,KVUEPP,NaN,Denise,O.,1969-09-24,58215,W,NaN,NaN,NaN,H007285161,Denise,Osman,1969-09-24,58215,W
2,59067,20212,KVUEPP,Q915251003,ALEXANDRA,Brinkmann,1992-09-18,44822,M,NaN,NaN,NaN,Q915251003,ALEXANDRA,Brinkmann,1992-09-18,49417,M
3,88797,20243,BEARBEITET,S086669189,NaN,Belik,1987-04-30,31707,M,NaN,NaN,NaN,S086669189,LAURA,Celik,1987-04-30,31707,M
4,75879,20241,BEARBEITET,X679432169,HORST,B.,1992-01-27,48272,W,1.0,NaN,2024-02-01,X679432169,Horst,Büttner,1992-01-27,48047,W
5,99882,20224,BEARBEITET,V078204633,Harald,Sahin,1977-06-12,42006,W,NaN,NaN,NaN,V078204633,Harald,Sahin,1977-06-12,57421,W
6,49279,20241,BEARBEITET,H549444795,LUISE,Günt.,2000-08-14,32031,W,NaN,NaN,NaN,H549444795,LUISE,Günther,2000-08-14,32031,W
7,95164,20203,UNBEARBEITET,I060694141,Johanna,WENZEL,1993-12-08,37880,M,2.0,NaN,2020-07-06,I060694141,Johanna,WENZEL,1993-12-08,37880,M
8,55196,20203,UNBEARBEITET,V865104216,Jan,Schroeder,1979-08-18,48206,W,NaN,NaN,NaN,V865104216,Jan,Schroeder,1979-08-18,48206,W
9,78354,20222,UNBEARBEITET,A836886113,Jens,Hofmann,1990-05-13,46250,W,2.0,NaN,2022-06-19,A836886113,Jens,Hofmann,1990-05-13,46250,W


In [3]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nEmpty strings per column:")
print((df == '').sum())


Missing values per column:
entity_id                            0
DWH_ZEITRAUM                         0
Datenkoerper                         0
EGKVERSICHERTENNUMMER            80159
VORNAME                          12450
NACHNAME                         12340
Geburtsdatum                      5089
PLZ                              38154
GESCHLECHT                       18597
ERGEBNISONLINEPRUEFUNG          210045
ERRORCODE                       464534
ONLINEPRUEFUNGSDATUM            246016
golden_EGKVERSICHERTENNUMMER         0
golden_VORNAME                       0
golden_NACHNAME                      0
golden_Geburtsdatum                  0
golden_PLZ                           0
golden_GESCHLECHT                    0
dtype: int64

Empty strings per column:
entity_id                       0
DWH_ZEITRAUM                    0
Datenkoerper                    0
EGKVERSICHERTENNUMMER           0
VORNAME                         0
NACHNAME                        0
Geburtsdatum               

## 2. Record Linkage Functions

Implementation of the cascading matching strategy with Union-Find for transitive closure.


In [4]:
class UnionFind:
    """Union-Find data structure for efficient grouping with path compression."""
    
    def __init__(self):
        self.parent = {}
        self.rank = {}
    
    def find(self, x: int) -> int:
        if x not in self.parent:
            self.parent[x] = x
            self.rank[x] = 0
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    
    def union(self, x: int, y: int) -> None:
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1


In [5]:
def normalize_string(s) -> str | None:
    """Normalize string for matching: uppercase and strip whitespace."""
    if pd.isna(s) or s == '':
        return None
    return str(s).strip().upper()


def create_matching_key(row: pd.Series, fields: list[str]) -> tuple | None:
    """Create a matching key from specified fields. Returns None if any field is missing."""
    values = []
    for field in fields:
        val = normalize_string(row.get(field))
        if val is None:
            return None
        values.append(val)
    return tuple(values)


In [6]:
def run_record_linkage(df: pd.DataFrame) -> pd.Series:
    """
    Run the complete record linkage algorithm.
    
    Matching phases:
    1. EGK + Geburtsdatum
    2. EGK + Vorname + Nachname
    3. Vorname + Nachname + Geburtsdatum + PLZ
    
    Returns predicted cluster IDs for each record.
    """
    uf = UnionFind()
    
    for idx in df.index:
        uf.find(idx)
    
    matching_rules = [
        ('EGK+DOB', ['EGKVERSICHERTENNUMMER', 'Geburtsdatum']),
        ('EGK+Name', ['EGKVERSICHERTENNUMMER', 'VORNAME', 'NACHNAME']),
        ('Name+DOB+PLZ', ['VORNAME', 'NACHNAME', 'Geburtsdatum', 'PLZ']),
    ]
    
    for rule_name, fields in matching_rules:
        key_to_indices = defaultdict(list)
        
        for idx, row in df.iterrows():
            key = create_matching_key(row, fields)
            if key is not None:
                key_to_indices[key].append(idx)
        
        unions_made = 0
        for indices in key_to_indices.values():
            if len(indices) > 1:
                first = indices[0]
                for other in indices[1:]:
                    if uf.find(first) != uf.find(other):
                        uf.union(first, other)
                        unions_made += 1
        
        print(f"  {rule_name}: {len(key_to_indices):,} keys, {unions_made:,} unions")
    
    return pd.Series([uf.find(idx) for idx in df.index], index=df.index)


## 3. Run Record Linkage


In [7]:
print("Running record linkage...\n")
df['predicted_cluster'] = run_record_linkage(df)

print(f"\nPredicted clusters: {df['predicted_cluster'].nunique():,}")
print(f"Ground truth entities: {df['entity_id'].nunique():,}")


Running record linkage...

  EGK+DOB: 108,248 keys, 307,825 unions
  EGK+Name: 183,926 keys, 3,597 unions
  Name+DOB+PLZ: 241,618 keys, 45,236 unions

Predicted clusters: 143,847
Ground truth entities: 100,000


## 4. Evaluation Metrics

Standard record linkage metrics:
- **Pairwise Precision**: Of all predicted matches, how many are correct?
- **Pairwise Recall**: Of all true matches, how many did we find?
- **F1 Score**: Harmonic mean of precision and recall


In [8]:
def count_pairs_in_clusters(labels: pd.Series) -> int:
    """Count total pairs within clusters."""
    counts = labels.value_counts()
    return int((counts * (counts - 1) / 2).sum())


def compute_pairwise_metrics(true_labels: pd.Series, pred_labels: pd.Series) -> dict:
    """Compute pairwise precision, recall, and F1."""
    true_clusters = defaultdict(set)
    pred_clusters = defaultdict(set)
    
    for idx, (true_id, pred_id) in enumerate(zip(true_labels, pred_labels)):
        true_clusters[true_id].add(idx)
        pred_clusters[pred_id].add(idx)
    
    # Count true positives
    tp = 0
    for pred_members in pred_clusters.values():
        true_id_counts = defaultdict(int)
        for idx in pred_members:
            true_id_counts[true_labels.iloc[idx]] += 1
        for count in true_id_counts.values():
            tp += count * (count - 1) // 2
    
    total_pred_pairs = count_pairs_in_clusters(pred_labels)
    total_true_pairs = count_pairs_in_clusters(true_labels)
    
    precision = tp / total_pred_pairs if total_pred_pairs > 0 else 0
    recall = tp / total_true_pairs if total_true_pairs > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'true_positives': tp,
        'predicted_pairs': total_pred_pairs,
        'true_pairs': total_true_pairs,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


In [9]:
print("Computing pairwise metrics...")
metrics = compute_pairwise_metrics(df['entity_id'], df['predicted_cluster'])

print(f"\n{'='*50}")
print("RECORD LINKAGE EVALUATION RESULTS")
print(f"{'='*50}")
print(f"\nPair Counts:")
print(f"  True pairs (ground truth):  {metrics['true_pairs']:,}")
print(f"  Predicted pairs:            {metrics['predicted_pairs']:,}")
print(f"  True positives:             {metrics['true_positives']:,}")
print(f"\nMetrics:")
print(f"  Precision: {metrics['precision']:.4f} ({metrics['precision']*100:.2f}%)")
print(f"  Recall:    {metrics['recall']:.4f} ({metrics['recall']*100:.2f}%)")
print(f"  F1 Score:  {metrics['f1']:.4f} ({metrics['f1']*100:.2f}%)")


Computing pairwise metrics...

RECORD LINKAGE EVALUATION RESULTS

Pair Counts:
  True pairs (ground truth):  1,202,726
  Predicted pairs:            1,004,721
  True positives:             1,004,721

Metrics:
  Precision: 1.0000 (100.00%)
  Recall:    0.8354 (83.54%)
  F1 Score:  0.9103 (91.03%)


## 6.1 Export Errors to Disk

Export all false positives and false negatives for deep analysis with foundational models.


In [10]:
def get_all_false_positives(df: pd.DataFrame) -> pd.DataFrame:
    """Get all records in clusters containing multiple different entities."""
    merged_clusters = df.groupby('predicted_cluster')['entity_id'].nunique()
    merged_cluster_ids = merged_clusters[merged_clusters > 1].index
    return df[df['predicted_cluster'].isin(merged_cluster_ids)].copy()


def get_all_false_negatives(df: pd.DataFrame) -> pd.DataFrame:
    """Get all records from entities that are split across multiple clusters."""
    split_entities = df.groupby('entity_id')['predicted_cluster'].nunique()
    split_entity_ids = split_entities[split_entities > 1].index
    return df[df['entity_id'].isin(split_entity_ids)].copy()


fp_df = get_all_false_positives(df)
fn_df = get_all_false_negatives(df)

print(f"False positives: {len(fp_df):,} records from {fp_df['predicted_cluster'].nunique():,} merged clusters")
print(f"False negatives: {len(fn_df):,} records from {fn_df['entity_id'].nunique():,} split entities")


False positives: 0 records from 0 merged clusters
False negatives: 196,387 records from 35,466 split entities


In [11]:
output_cols = [
    'entity_id', 'predicted_cluster', 
    'EGKVERSICHERTENNUMMER', 'VORNAME', 'NACHNAME', 'Geburtsdatum', 'PLZ', 'GESCHLECHT',
    'golden_EGKVERSICHERTENNUMMER', 'golden_VORNAME', 'golden_NACHNAME', 
    'golden_Geburtsdatum', 'golden_PLZ', 'golden_GESCHLECHT'
]

fp_df[output_cols].to_csv('false_positives.csv', index=False)
fn_df[output_cols].to_csv('false_negatives.csv', index=False)

print(f"Saved: false_positives.csv ({len(fp_df):,} rows)")
print(f"Saved: false_negatives.csv ({len(fn_df):,} rows)")


Saved: false_positives.csv (0 rows)
Saved: false_negatives.csv (196,387 rows)


## 5. Cluster-Level Analysis


In [12]:
def analyze_clusters(df: pd.DataFrame) -> dict:
    """Analyze cluster quality at the cluster level."""
    results = {'perfect': 0, 'split': 0, 'merged': 0}
    
    for pred_id, group in df.groupby('predicted_cluster'):
        true_ids = group['entity_id'].unique()
        if len(true_ids) == 1:
            entity_id = true_ids[0]
            total_records = (df['entity_id'] == entity_id).sum()
            if len(group) == total_records:
                results['perfect'] += 1
            else:
                results['split'] += 1
        else:
            results['merged'] += 1
    
    return results


cluster_analysis = analyze_clusters(df)
total_predicted = df['predicted_cluster'].nunique()

print(f"\nCluster-Level Analysis:")
print(f"  Total predicted clusters: {total_predicted:,}")
print(f"  Perfect clusters:  {cluster_analysis['perfect']:,} ({cluster_analysis['perfect']/total_predicted*100:.1f}%)")
print(f"  Split clusters:    {cluster_analysis['split']:,} ({cluster_analysis['split']/total_predicted*100:.1f}%)")
print(f"  Merged clusters:   {cluster_analysis['merged']:,} ({cluster_analysis['merged']/total_predicted*100:.1f}%)")



Cluster-Level Analysis:
  Total predicted clusters: 143,847
  Perfect clusters:  64,534 (44.9%)
  Split clusters:    79,313 (55.1%)
  Merged clusters:   0 (0.0%)


## 6. Error Analysis

Examine examples of false positives (incorrectly merged) and false negatives (incorrectly split).


In [13]:
def find_false_positives(df: pd.DataFrame, n: int = 3) -> pd.DataFrame:
    """Find examples of incorrectly merged records (different entities in same cluster)."""
    examples = []
    for pred_id, group in df.groupby('predicted_cluster'):
        if len(group['entity_id'].unique()) > 1:
            examples.append(group.head(6))
            if len(examples) >= n:
                break
    return pd.concat(examples) if examples else pd.DataFrame()


def find_false_negatives(df: pd.DataFrame, n: int = 3) -> pd.DataFrame:
    """Find examples of incorrectly split records (same entity in different clusters)."""
    examples = []
    for entity_id, group in df.groupby('entity_id'):
        if len(group['predicted_cluster'].unique()) > 1:
            examples.append(group.head(6))
            if len(examples) >= n:
                break
    return pd.concat(examples) if examples else pd.DataFrame()


In [14]:
cols = ['entity_id', 'predicted_cluster', 'EGKVERSICHERTENNUMMER', 'VORNAME', 'NACHNAME', 'Geburtsdatum', 'PLZ']

print("=" * 80)
print("FALSE POSITIVES (Different Entities Incorrectly Merged)")
print("=" * 80)
fp_examples = find_false_positives(df, n=3)
if len(fp_examples) > 0:
    display(fp_examples[cols])
else:
    print("No false positives found!")


FALSE POSITIVES (Different Entities Incorrectly Merged)
No false positives found!


In [15]:
print("=" * 80)
print("FALSE NEGATIVES (Same Entity Incorrectly Split)")
print("=" * 80)
fn_examples = find_false_negatives(df, n=3)
if len(fn_examples) > 0:
    display(fn_examples[cols])
else:
    print("No false negatives found!")


FALSE NEGATIVES (Same Entity Incorrectly Split)


,entity_id,predicted_cluster,EGKVERSICHERTENNUMMER,VORNAME,NACHNAME,Geburtsdatum,PLZ
84086,1,84086,B018451466,RENATE,SCHRÖDER,1981-06-09,44704
185089,1,84086,B018451466,renate,S.,1981-06-09,44704
253825,1,84086,B018451466,RENATE,SCHRÖDER,1981-06-09,44704
295719,1,295719,NaN,NaN,SCHRÖDER,1981-06-09,NaN
389098,1,84086,B018451466,RENATE,SCHRÖDER,1981-06-09,44704
448014,1,448014,NaN,RENATE,SCHRÖDER,1381-06-09,4470
129161,3,129161,NaN,JESSICA,WEBER,1935-11-08,45841
289874,3,289874,NaN,JESSIIA,WEBERR,1935-11-08,45841
362303,3,362303,Z542784987,JESS.,WEBER,1935-11-08,45841
32127,8,32127,B726284987,Maja,FRIEDRICH,1993-02-19,47769


## 7. Per-Rule Coverage Analysis


In [16]:
def analyze_rule_coverage(df: pd.DataFrame) -> pd.DataFrame:
    """Analyze coverage of each matching rule."""
    rules = [
        ('EGK+DOB', ['EGKVERSICHERTENNUMMER', 'Geburtsdatum']),
        ('EGK+Name', ['EGKVERSICHERTENNUMMER', 'VORNAME', 'NACHNAME']),
        ('Name+DOB+PLZ', ['VORNAME', 'NACHNAME', 'Geburtsdatum', 'PLZ']),
    ]
    
    results = []
    for rule_name, fields in rules:
        valid = df[fields].notna().all(axis=1) & (df[fields] != '').all(axis=1)
        valid_count = valid.sum()
        
        if valid_count > 0:
            keys = df.loc[valid, fields].apply(
                lambda row: tuple(str(v).upper() for v in row), axis=1
            )
            key_counts = keys.value_counts()
            linking_keys = (key_counts > 1).sum()
            linked_records = key_counts[key_counts > 1].sum()
        else:
            linking_keys, linked_records = 0, 0
        
        results.append({
            'Rule': rule_name,
            'Valid Records': valid_count,
            'Coverage %': f"{valid_count / len(df) * 100:.1f}%",
            'Linking Keys': linking_keys,
            'Linked Records': linked_records,
        })
    
    return pd.DataFrame(results)


print("\nMatching Rule Coverage:")
display(analyze_rule_coverage(df))



Matching Rule Coverage:


,Rule,Valid Records,Coverage %,Linking Keys,Linked Records
0,EGK+DOB,416073,83.1%,93402,401227
1,EGK+Name,400118,79.9%,79867,257203
2,Name+DOB+PLZ,436898,87.3%,84063,242072


## 8. Summary


In [17]:
print("\n" + "=" * 60)
print("RECORD LINKAGE MODEL VALIDATION SUMMARY")
print("=" * 60)
print(f"\nDataset:")
print(f"  Total records:           {len(df):,}")
print(f"  Ground truth entities:   {df['entity_id'].nunique():,}")
print(f"  Predicted clusters:      {df['predicted_cluster'].nunique():,}")
print(f"\nPairwise Metrics:")
print(f"  Precision: {metrics['precision']*100:.2f}%")
print(f"  Recall:    {metrics['recall']*100:.2f}%")
print(f"  F1 Score:  {metrics['f1']*100:.2f}%")
print(f"\nCluster Quality:")
print(f"  Perfect matches: {cluster_analysis['perfect']:,} clusters")
print(f"  Over-merged:     {cluster_analysis['merged']:,} clusters (false positives)")
print(f"  Under-linked:    {cluster_analysis['split']:,} clusters (false negatives)")
print("\n" + "=" * 60)



RECORD LINKAGE MODEL VALIDATION SUMMARY

Dataset:
  Total records:           500,505
  Ground truth entities:   100,000
  Predicted clusters:      143,847

Pairwise Metrics:
  Precision: 100.00%
  Recall:    83.54%
  F1 Score:  91.03%

Cluster Quality:
  Perfect matches: 64,534 clusters
  Over-merged:     0 clusters (false positives)
  Under-linked:    79,313 clusters (false negatives)

